# 自然言語処理（NLP）

自然言語処理では、文字列をモデルが扱える数値列へ変換します。最初に決めるのは、文章をどの単位に分け、どの整数 ID として渡し、どの位置から損失を取るかです。分かち書き、語彙、ID 化、埋め込み、系列の集約、次トークン予測、損失マスクが入力から学習信号までの流れを作ります。

日本語は空白で単語境界が示されないため、単純な空白分割だけでは壊れやすくなります。文字単位、部分文字列、辞書ベースの分割を目的に応じて選び、未知語と系列長のバランスを取ります。

## 文字列をトークン列へ変える

同じ文でも、空白分割、文字分割、辞書ベース分割で系列長と語彙の見え方が変わります。分割の単位はモデルの入力そのものです。

In [ ]:
import math
from collections import Counter, defaultdict

texts = [
    '自然言語処理は文章を数値列に変換する',
    '言語モデルは文脈から次のトークンを予測する',
    '検索拡張では根拠文を入力へ追加する',
    '分類モデルは文章の意図や感情を判定する',
]

lexicon = ['自然言語処理', '言語モデル', '検索拡張', '分類モデル', '文章', '数値列', '文脈', '次', 'トークン', '予測', '根拠文', '入力', '追加', '意図', '感情', '判定', '変換']

def whitespace_tokenize(text):
    return text.split()

def char_tokenize(text):
    return list(text)

def lexicon_tokenize(text, words):
    tokens = []
    i = 0
    words = sorted(words, key=len, reverse=True)
    while i < len(text):
        match = None
        for word in words:
            if text.startswith(word, i):
                match = word
                break
        if match is None:
            tokens.append(text[i])
            i += 1
        else:
            tokens.append(match)
            i += len(match)
    return tokens

for text in texts[:2]:
    print(text)
    print(' whitespace:', whitespace_tokenize(text))
    print(' char      :', char_tokenize(text)[:12], 'len=', len(char_tokenize(text)))
    print(' lexicon   :', lexicon_tokenize(text, lexicon))

## 語彙と ID 変換

ニューラルネットは文字列を直接受け取りません。トークンを語彙表で整数 ID に変換します。未知トークンと padding の扱いも、この段階で決めます。

In [ ]:
tokenized = [lexicon_tokenize(text, lexicon) for text in texts]
counter = Counter(token for row in tokenized for token in row)
vocab = ['<pad>', '<unk>'] + [token for token, _ in counter.most_common()]
stoi = {token: i for i, token in enumerate(vocab)}
itos = {i: token for token, i in stoi.items()}

def encode(tokens):
    return [stoi.get(token, stoi['<unk>']) for token in tokens]

def decode(ids):
    return [itos.get(i, '<unk>') for i in ids]

encoded = [encode(row) for row in tokenized]
print('vocab size:', len(vocab))
print('tokens:', tokenized[0])
print('ids   :', encoded[0])
print('unknown:', encode(['未知語', '文章']))

## padding と attention mask

ミニバッチでは系列長をそろえます。padding は計算上の穴埋めなので、attention mask や loss mask で無視します。mask を忘れると、存在しない穴埋めトークンまで文脈や損失として扱ってしまいます。

In [ ]:
def pad_batch(sequences, pad_id=0):
    max_len = max(len(seq) for seq in sequences)
    padded = []
    mask = []
    for seq in sequences:
        extra = max_len - len(seq)
        padded.append(seq + [pad_id] * extra)
        mask.append([1] * len(seq) + [0] * extra)
    return padded, mask

padded, attention_mask = pad_batch(encoded)
print('padded')
for row in padded:
    print(row)
print('mask')
for row in attention_mask:
    print(row)

## 埋め込みは表引きから始まる

Embedding は token ID からベクトルを引く表です。学習が進むと、似た文脈で使われるトークンが近いベクトルになります。小さな固定ベクトルで文章表現を作ります。

In [ ]:
def make_embedding_table(size, dim):
    table = []
    for i in range(size):
        table.append([math.sin((i + 1) * (j + 1)) * 0.4 + math.cos(i + j + 1) * 0.2 for j in range(dim)])
    return table

embeddings = make_embedding_table(len(vocab), dim=5)

def mean_pool(ids, mask):
    vectors = [embeddings[i] for i, m in zip(ids, mask) if m]
    return [sum(vec[d] for vec in vectors) / len(vectors) for d in range(len(vectors[0]))]

def rounded(vec):
    return [round(x, 3) for x in vec]

sentence_vectors = [mean_pool(ids, mask) for ids, mask in zip(padded, attention_mask)]
for text, vec in zip(texts, sentence_vectors):
    print(text, rounded(vec))

## 文書分類を手計算で作る

文章表現ができると、分類器を上に置けます。小さなロジスティック回帰を学習し、NLP の基本形である「エンコードして分類する」流れを確認します。

In [ ]:
train_texts = [
    ('配送が遅いので問い合わせたい', 'support'),
    ('返金の手続きを知りたい', 'support'),
    ('新しい料金プランを教えて', 'sales'),
    ('法人契約の見積もりがほしい', 'sales'),
    ('ログインできないので助けて', 'support'),
    ('有料プランの違いを知りたい', 'sales'),
]
keywords = ['配送', '返金', '問い合わせ', 'ログイン', '料金', 'プラン', '法人', '見積もり', '契約', '助け']

def keyword_features(text):
    return [1.0 if word in text else 0.0 for word in keywords] + [len(text) / 30]

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def train_binary_classifier(rows, steps=240, lr=0.4):
    weights = [0.0] * (len(keyword_features(rows[0][0])) + 1)
    for _ in range(steps):
        for text, label in rows:
            x = [1.0] + keyword_features(text)
            y = 1.0 if label == 'sales' else 0.0
            pred = sigmoid(sum(w * v for w, v in zip(weights, x)))
            err = pred - y
            for i in range(len(weights)):
                weights[i] -= lr * err * x[i]
    return weights

def predict_label(weights, text):
    x = [1.0] + keyword_features(text)
    prob = sigmoid(sum(w * v for w, v in zip(weights, x)))
    return ('sales' if prob >= 0.5 else 'support', prob)

clf = train_binary_classifier(train_texts)
for text in ['料金プランの見積もりがほしい', 'ログインの問い合わせをしたい']:
    label, prob = predict_label(clf, text)
    print(text, label, round(prob, 3))

## 次トークン予測

言語モデルは、左側の文脈から次のトークン分布を出します。n-gram を使うと、ニューラルネットなしでも「文脈から次を予測する」形を確認できます。

In [ ]:
corpus = [
    list('言語モデルは次を予測する'),
    list('言語モデルは文脈を見る'),
    list('分類モデルは意図を判定する'),
    list('検索拡張は根拠を見る'),
]

bigram = defaultdict(Counter)
for seq in corpus:
    for a, b in zip(seq, seq[1:]):
        bigram[a][b] += 1

def next_token_distribution(prev, temperature=1.0):
    counts = bigram[prev]
    if not counts:
        return []
    scores = {tok: count ** (1 / temperature) for tok, count in counts.items()}
    total = sum(scores.values())
    return sorted([(tok, score / total) for tok, score in scores.items()], key=lambda x: x[1], reverse=True)

for prev in ['は', 'モ', 'を']:
    print(prev, [(tok, round(prob, 3)) for tok, prob in next_token_distribution(prev)])

## クロスエントロピー損失

モデルが正解トークンへ高い確率を置くほど損失は小さくなります。softmax と negative log likelihood が次トークン予測の基本損失です。

In [ ]:
def softmax(logits):
    m = max(logits)
    exps = [math.exp(x - m) for x in logits]
    total = sum(exps)
    return [x / total for x in exps]

def cross_entropy(logits, target_index):
    probs = softmax(logits)
    return -math.log(probs[target_index]), probs

logits = [0.2, 1.4, -0.3, 0.8]
target = 1
loss, probs = cross_entropy(logits, target)
print('probs:', rounded(probs))
print('loss:', round(loss, 3))

## SFT の loss mask

指示と回答を 1 本の系列にしても、学習したいのは回答部分です。loss mask によって、どの位置に損失を掛けるかを指定します。

In [ ]:
instruction = '質問: ベルマン方程式を一文で説明してください。'
answer = '回答: 価値を報酬と次状態価値で再帰的に表す式です。'
sequence = list(instruction + answer)
answer_start = len(instruction)
loss_mask = [0] * answer_start + [1] * len(answer)

for ch, mask in list(zip(sequence, loss_mask))[answer_start - 5:answer_start + 12]:
    print(ch, mask)
print('loss positions:', sum(loss_mask), 'of', len(loss_mask))

## attention は文脈を重み付きで読む

Self-attention は、各位置がどのトークンを参照するかを重みで表します。分類でも生成でも、重要な文脈を選んで混ぜる操作が中核になります。

In [ ]:
tokens = ['検索', '根拠', '回答']
vecs = {
    '検索': [0.9, 0.1, 0.0],
    '根拠': [0.8, 0.2, 0.1],
    '回答': [0.1, 0.7, 0.6],
}

def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

def attend(query_token, key_tokens):
    query = vecs[query_token]
    scores = [dot(query, vecs[token]) / math.sqrt(len(query)) for token in key_tokens]
    weights = softmax(scores)
    context = [sum(w * vecs[token][d] for w, token in zip(weights, key_tokens)) for d in range(3)]
    return weights, context

weights, context = attend('回答', tokens)
print('weights:', list(zip(tokens, rounded(weights))))
print('context:', rounded(context))

## LoRA のパラメータ削減

LoRA は大きな重み行列全体を更新せず、低ランク行列だけを学習します。更新対象を絞ることで、メモリと学習コストを下げます。

In [ ]:
def lora_param_count(d_in, d_out, rank):
    full = d_in * d_out
    lora = rank * (d_in + d_out)
    return full, lora, lora / full

for rank in [4, 8, 16, 32]:
    full, lora, ratio = lora_param_count(4096, 4096, rank)
    print(f'rank={rank:2d} full={full:,} lora={lora:,} ratio={ratio:.4f}')

## 実務で確認する点

NLP の品質は、モデル構造だけで決まりません。トークン化、語彙、未知語、padding、loss mask、系列長、文脈の選び方、評価データの作り方が同じくらい重要です。

分類では入力全体を安定した表現に集約します。生成では位置ごとに次トークンを当て、SFT では回答部分だけに損失を掛けます。どの位置に情報を入れ、どの位置から学習信号を取るかが設計の中心です。